# Notebook Overview

Purpose:

*   Transform Bronze product data into a standardized Silver dataset.
*   Validate product records against documented data-quality rules.
*   Separate accepted and quarantined records.
*   Persist outputs and reconcile quality metrics.

### Configuration / Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

FILE_PATH = "/content/drive/MyDrive/pyspark-ecommerce-analytics-data"
BRONZE_PRODUCTS_PATH = f"{FILE_PATH}/bronze/products"
QUARANTINE_PRODUCTS_PATH = f"{FILE_PATH}/quarantine/products"
SILVER_PRODUCTS_PATH = f"{FILE_PATH}/silver/products"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = SparkSession.builder.appName('product-silver').getOrCreate()
spark.version

'4.0.4'

### Load Data from Bronze

In [ ]:
def load_data(FILE_PATH):
  try:
    df_product = spark.read.parquet(FILE_PATH).withColumn(
        '_source_file', F.lit(F.input_file_name())
    )
    print('Data Load Successful')
    return df_product

  except Exception as e:
    raise RuntimeError(f"Data Load Failed: {str(e)}")

df_product = load_data(BRONZE_PRODUCTS_PATH)

Data Load Successful


### Row Count and Bronze Product Schema

In [ ]:
def schema_and_count(df):
  print('\nRow Count:', df.count())
  print('Schema')
  print('======')
  df.printSchema()

schema_and_count(df_product)


Row Count: 1000
Schema
root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- cost_price: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = false)



### Validate the Bronze Schema

In [ ]:
def validate_products_schema(df):
  try:
    expected_columns = ['brand',
                        'category',
                        'cost_price',
                        'created_date',
                        'is_active',
                        'product_id',
                        'product_name',
                        'subcategory',
                        'unit_price',
                        '_ingestion_timestamp',
                        '_source_file'
                        ]

    df_column = df.columns

    missing_cols = list(set(expected_columns) - set(df_column))

    expected_schema = {
          'brand': 'string',
          'category': 'string',
          'cost_price': 'string',
          'created_date': 'string',
          'is_active': 'string',
          'product_id': 'string',
          'product_name': 'string',
          'subcategory': 'string',
          'unit_price' : 'string',
          '_ingestion_timestamp': 'timestamp',
          '_source_file': 'string'
      }

    actual_types = dict(df.dtypes)

    type_missmatch = []
    for col, expected_type in expected_schema.items():
      if col in actual_types and actual_types[col] != expected_type:
        type_missmatch.append(f"Column '{col}' expected '{expected_type}', but got '{actual_types[col]}'")

    error_messages = []
    if missing_cols:
      error_messages.append(f'Missing_cols: {missing_cols}')
    if type_missmatch:
      error_messages.append(f'Type Missmatch: {type_missmatch}')

    if error_messages:
      raise ValueError("Product Schema Validation Failed:\n" + "\n".join(error_messages))
    print("Schema Validation passed successfully!")

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'validate_products_schema': {str(e)}") from e

### Preserve Audit Information

In [ ]:
def preserve_audit(df):
  df_preserve = df
  df_preserve = df.withColumn(
      '_original_created_date',
      F.col('created_date')
      ).withColumn(
          '_original_unit_price',
          F.col('unit_price')
      ).withColumn(
          '_original_cost_price',
          F.col('cost_price')
      )

  return df_preserve

### Standardize String Columns

In [ ]:
def standardize_product_strings(df):
  try:
    string_cols = [
        'brand',
        'category',
        'cost_price',
        'created_date',
        'is_active',
        'product_id',
        'product_name',
        'subcategory',
        'unit_price',
        '_source_file'
      ]

    df_clean = df

    for c in string_cols:
      df_clean = df_clean.withColumn(
          c,
          F.when(F.trim(F.lower(F.col(c))).isin('n/a', 'na', 'unknown', 'null', ''), None).otherwise(F.trim(F.col(c)))
      )

    return df_clean
  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'standardize_product_strings': {str(e)}") from e

### Cast Types

In [ ]:
from pyspark.sql.types import DecimalType, DateType

def cast_product_types(df):
  try:
    if 'cost_price' in df.columns:
      df_typed = df.withColumn('cost_price', F.col('cost_price').try_cast(DecimalType(10,2)))

    if 'created_date' in df.columns:
      df_typed = df_typed.withColumn('created_date', F.col('created_date').try_cast(DateType()))

    if 'unit_price' in df.columns:
      df_typed = df_typed.withColumn('unit_price', F.col('unit_price').try_cast(DecimalType(10,2)))

    return df_typed

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'cast_product_types': {str(e)}") from e

### Validate Product Records

In [ ]:
def validate_products_records(df):
  try:
    electronic_subcategory = ['accessories', 'phones', 'audio', 'laptops']
    df_validated = df.withColumn(
        'is_valid',
        F.when(F.col('product_id').isNull(), 'Missing or NULL Product ID')

        .when(F.col('created_date').isNull(), 'Invalid or NULL Date')

        .when((F.col('cost_price').isNull()) | (F.col('cost_price') == 0) | (F.col('cost_price') < 0), 'Missing or Invalid Cost Price')

        .when((F.col('unit_price').isNull()) | (F.col('unit_price') == 0) | (F.col('unit_price') < 0), 'Missing or Invalid Unit Price')

        )

    df_validated = df_validated.withColumn(
        'category',
        F.when(F.lower(F.col('subcategory')).isin(electronic_subcategory), 'Electronics').otherwise(F.col('category'))

    )

    return df_validated

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'validate_products_records': {str(e)}") from e

### Handle Duplicate Product IDs

In [ ]:
from pyspark.sql import Window as W

def handle_duplicate_product_ids(df):
  try:
    window_spec = W.partitionBy('product_id').orderBy(F.col('_ingestion_timestamp').desc())

    df_ranked = df.withColumn(
        'rank',
        F.row_number().over(window_spec)
    )

    df_mark = df_ranked.withColumn(
        'is_valid',
        F.coalesce(
            F.col('is_valid'),
            F.when(F.col('rank') > 1, 'Duplicate Product ID')
        )
    ).drop('rank')

    return df_mark

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'handle_duplicate_order_ids': {str(e)}") from e

### Split Valid and Quarantined Records

In [ ]:
def split_products_by_quality(df):
  try:
    df_valid = df.filter(F.col('is_valid').isNull()).drop('is_valid', '_original_created_date', '_original_unit_price', '_original_cost_price')

    df_quarantine = df.filter(F.col('is_valid').isNotNull())

    return df_valid, df_quarantine

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'split_products_by_quality': {str(e)}") from e

### Orchestrate the Products Transformation

In [ ]:
def transform_products(df):
  try:
    print('===============================')
    print('TRANSFORMING DATA')
    print('===============================')
    print('Start: Validating Bronze Schema')
    validate_products_schema(df)
    print('Finished: Validaing Bronze Schema')
    print('Start: Preserving Data for Audit')
    preserved_df = preserve_audit(df)
    print('Finished: Preserving Data for Audit')
    print('Start: Standardizing String Columns')
    standardize_df = standardize_product_strings(preserved_df)
    print('Finished: Standardizing String Columns')
    print('Start: Cast type Column')
    casted_df = cast_product_types(standardize_df)
    print('Finished: Cast type Column')
    print('Start: Validating Product Record')
    validated_df = validate_products_records(casted_df)
    print('Finished: Validating Product Record')
    print('Start: Handling and Marking duplicate id')
    df_mark = handle_duplicate_product_ids(validated_df)
    print('Finished: Handling and Marking duplicate id')
    print('Spiliting and Returning Validated and Quanrantine data')
    print('===============================')
    print('TRANSFORMING COMPLETE')
    print('===============================')
    return split_products_by_quality(df_mark)

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Failed inside 'transform_products': {str(e)}") from e

### **Transforming Product Data**

In [ ]:
valid_df, quarantine_df = transform_products(df_product)

TRANSFORMING DATA
Start: Validating Bronze Schema
Schema Validation passed successfully!
Finished: Validaing Bronze Schema
Start: Preserving Data for Audit
Finished: Preserving Data for Audit
Start: Standardizing String Columns
Finished: Standardizing String Columns
Start: Cast type Column
Finished: Cast type Column
Start: Validating Product Record
Finished: Validating Product Record
Start: Handling and Marking duplicate id
Finished: Handling and Marking duplicate id
Spiliting and Returning Validated and Quanrantine data
TRANSFORMING COMPLETE


### Calculate quality metrics and reconcile

In [ ]:
def generate_pipeline_metrics(df_bronze, df_valid, df_quarantine):
    print("\n" + "=" * 65)
    print(" PIPELINE RECONCILIATION & QUALITY METRICS REPORT")
    print("=" * 65)

    input_count = df_bronze.count()
    silver_count = df_valid.count()
    quarantine_count = df_quarantine.count()

    print("\n[1] RECONCILIATION SUMMARY")
    print(f"  • Total Bronze Input Count      : {input_count:,}")
    print(f"  • Accepted Silver Count         : {silver_count:,}")
    print(f"  • Quarantined Count             : {quarantine_count:,}")

    sum_outputs = silver_count + quarantine_count
    is_balanced = (input_count == sum_outputs)
    print(f"  • Reconciliation Balance Check  : {'PASSED (Match)' if is_balanced else 'FAILED (Mismatch!)'}")
    if not is_balanced:
        print(f"    WARNING: Input ({input_count:,}) != Silver ({silver_count:,}) + Quarantine ({quarantine_count:,})")

    print("\n[2] QUARANTINE FAILURE BREAKDOWN (Validation & Duplicates)")
    if quarantine_count > 0:
        print("  Breakdown of reasons records were quarantined:")
        df_quarantine.groupBy("is_valid").count().orderBy(F.col("count").desc()).show(truncate=False)
    else:
        print("  • No quarantined records found. 100% clean dataset!")

    print("\n[3] NULL COUNTS IN SILVER LAYER")
    print("  (Tracking optional profile fields alongside required ones)")
    null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_valid.columns]
    df_valid.select(null_exprs).show(truncate=False)

    print("\n[4] UNIQUENESS CHECK (product_id in Silver)")
    distinct_orders = df_valid.select("product_id").distinct().count()
    is_unique = (silver_count == distinct_orders)
    print(f"  • Total Silver Rows             : {silver_count:,}")
    print(f"  • Unique Product IDs           : {distinct_orders:,}")
    print(f"  • Uniqueness Constraint         : {'PASSED (No duplicate IDs in Silver)' if is_unique else 'FAILED (Duplicates present!)'}")

    print("=" * 65)

generate_pipeline_metrics(df_product, valid_df, quarantine_df)


 PIPELINE RECONCILIATION & QUALITY METRICS REPORT

[1] RECONCILIATION SUMMARY
  • Total Bronze Input Count      : 1,000
  • Accepted Silver Count         : 996
  • Quarantined Count             : 4
  • Reconciliation Balance Check  : PASSED (Match)

[2] QUARANTINE FAILURE BREAKDOWN (Validation & Duplicates)
  Breakdown of reasons records were quarantined:
+-----------------------------+-----+
|is_valid                     |count|
+-----------------------------+-----+
|Missing or Invalid Unit Price|1    |
|Missing or Invalid Cost Price|1    |
|Invalid or NULL Date         |1    |
|Duplicate Product ID         |1    |
+-----------------------------+-----+


[3] NULL COUNTS IN SILVER LAYER
  (Tracking optional profile fields alongside required ones)
+-----+--------+----------+------------+---------+----------+------------+-----------+----------+--------------------+------------+
|brand|category|cost_price|created_date|is_active|product_id|product_name|subcategory|unit_price|_ingestion_ti

### Adding Metadata

In [ ]:
from pyspark.sql.types import TimestampType


# Adding metadata to Quarantine Dataset
quarantine_df = quarantine_df.withColumn('_silver_processed_timestamp', F.current_timestamp())

# Adding metadata to Valid Dataset
valid_df = valid_df.withColumn('_silver_processed_timestamp', F.current_timestamp())

silver_product_df = valid_df.select(
    'product_id',
    'product_name',
    'created_date',
    'brand',
    'category',
    'subcategory',
    'is_active',
    'cost_price',
    'unit_price',
    '_silver_processed_timestamp',
    '_source_file'
)

silver_product_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- cost_price: decimal(10,2) (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = false)
 |-- _source_file: string (nullable = true)



### ***Write into Silver***

In [ ]:
try:
  print('INFO: Writing records to Quarantine layer...')
  # Write Quarantine Data
  quarantine_df.write.mode('overwrite').parquet(QUARANTINE_PRODUCTS_PATH)
  print('SUCCESS: Quarantine dataset written successfully.')

  print('INFO: Writing clean records to Silver layer...')
  # Write Siver Data into Layer
  silver_product_df.write.mode('overwrite').parquet(SILVER_PRODUCTS_PATH)
  print('SUCCESS: Silver dataset written successfully.')

except Exception as e:
    raise RuntimeError(f"[Pipeline Error] Failed to write Silver dataset: {str(e)}") from e


INFO: Writing records to Quarantine layer...
SUCCESS: Quarantine dataset written successfully.
INFO: Writing clean records to Silver layer...
SUCCESS: Silver dataset written successfully.


### Read Back and Verify

In [ ]:
df = spark.read.parquet(SILVER_PRODUCTS_PATH)

quarantine_df = spark.read.parquet(QUARANTINE_PRODUCTS_PATH)

schema_and_count(df)
schema_and_count(quarantine_df)


Row Count: 996
Schema
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- cost_price: decimal(10,2) (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)


Row Count: 4
Schema
root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- cost_price: decimal(10,2) (nullable = true)
 |-- created_date: date (nullable = true)
 |-- is_active: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _sour